# Bronze: Airports (Static Reference data)
- **Source:  `abfss://bronze@.../airports/airports.csv` (uploaded by Airflow)**
- **Target**: Delta table at `abfss://bronze@.../airports_delta/`

One time batch read. Converts CSV to Delta, filters to relevant fields.

## Imports

In [0]:
email = dbutils.secrets.get(scope="flight-delay", key="db-email")

In [0]:
import sys
sys.path.insert(0, f"/Workspace/Users/{email}/ml-final-project/databricks")

from common.config import (
    AIRPORTS_CSV_PATH,
    BRONZE_AIRPORTS_PATH,
    configure_storage_auth,
)

from pyspark.sql import functions as F

## Auth into ADLS

In [0]:
configure_storage_auth(spark)

## Read CSV

In [0]:
airports_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("escape", '"')
    .csv(AIRPORTS_CSV_PATH)
)

print(f"Rows read: {airports_raw.count():,}")
airports_raw.printSchema()

## Minimal Bronze Layer Transformations
Adding ingestion metadata

In [0]:
bronze_airports = (
    airports_raw
    .withColumn("ingestion_ts_utc", F.current_timestamp())
    .withColumn("source", F.lit("ourairports.com"))
)

## Write to Delta using Overwrite 
Because static data

In [0]:
(
    bronze_airports.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(BRONZE_AIRPORTS_PATH)
)

### Register as Delta table in UC

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS bronze.airports
    USING DELTA
    LOCATION '{BRONZE_AIRPORTS_PATH}'
""")

### Test

In [0]:
row_count = spark.read.format("delta").load(BRONZE_AIRPORTS_PATH).count()
print(f"Total rows in bronze.airports: {row_count:,}")

## Exit the Notebok

In [0]:
dbutils.notebook.exit(f'{{"rows_in_bronze": {row_count}}}')